# Causal Discovery with tetrad-port

This tutorial demonstrates the three causal discovery algorithms available in tetrad-port:

1. **PC** — Constraint-based algorithm (conditional independence tests)
2. **FGES** — Score-based algorithm (BIC scoring, greedy search)
3. **GFCI** — Hybrid algorithm (handles latent/unmeasured confounders)

All algorithms are implemented in C++ (ported from CMU's Tetrad library) and exposed via Python bindings.

In [ ]:
import numpy as np
import pandas as pd
from tetrad_port import TetradPort

tp = TetradPort()

## Create Sample Data

We generate data from a known causal structure: **X → Y → Z** (a chain) with an unobserved latent variable **L** that confounds **X** and **W**.

True graph: `L → X`, `L → W`, `X → Y`, `Y → Z`

Since **L** is unobserved, algorithms that assume causal sufficiency (PC, FGES) may produce spurious edges between X and W, while GFCI can detect the latent confounding.

In [ ]:
np.random.seed(42)
n = 3000

# Latent common cause (unobserved)
L = np.random.randn(n)

# Observed variables
X = 0.8 * L + 0.5 * np.random.randn(n)
W = 0.7 * L + 0.5 * np.random.randn(n)
Y = 0.6 * X + 0.5 * np.random.randn(n)
Z = 0.6 * Y + 0.5 * np.random.randn(n)

# Only include observed variables
df = pd.DataFrame({"X": X, "W": W, "Y": Y, "Z": Z})
print(f"Data shape: {df.shape}")
df.head()

## 1. PC Algorithm (Constraint-Based)

PC uses conditional independence tests (Fisher Z) to discover the causal skeleton, then orients edges. It assumes **causal sufficiency** — no latent confounders.

In [ ]:
pc_results, pc_graph = tp.run_pc(df, alpha=0.05)

print("PC Algorithm Results")
print(f"Nodes: {pc_results['nodes']}")
print(f"Edges ({pc_results['num_edges']}):")
for e in pc_results["edges"]:
    print(f"  {e}")
print(f"\nDirected: {pc_graph['directed_edges']}")
print(f"Undirected: {pc_graph['undirected_edges']}")

## 2. FGES Algorithm (Score-Based)

FGES (Fast Greedy Equivalence Search) uses BIC scoring with a greedy forward-backward strategy. It also assumes causal sufficiency but is often faster than PC for large, sparse graphs. The `penalty_discount` parameter controls sparsity — higher values yield sparser graphs.

In [ ]:
fges_results, fges_graph = tp.run_fges(df, penalty_discount=1.0)

print("FGES Algorithm Results")
print(f"Nodes: {fges_results['nodes']}")
print(f"Model score: {fges_results['model_score']:.2f}")
print(f"Edges ({fges_results['num_edges']}):")
for e in fges_results["edges"]:
    print(f"  {e}")
print(f"\nDirected: {fges_graph['directed_edges']}")
print(f"Undirected: {fges_graph['undirected_edges']}")

## 3. GFCI Algorithm (Hybrid — Handles Latent Confounders)

GFCI combines FGES (score-based initial search) with FCI orientation rules to produce a **PAG (Partial Ancestral Graph)**. Unlike PC and FGES, GFCI does **not** assume causal sufficiency — it can detect latent common causes.

PAG edge types:
- `-->` directed (causal)
- `<->` bidirected (latent common cause)
- `o->` partially oriented
- `o-o` fully ambiguous

In [ ]:
gfci_results, gfci_graph = tp.run_gfci(df, alpha=0.05, penalty_discount=1.0)

print("GFCI Algorithm Results")
print(f"Nodes: {gfci_results['nodes']}")
print(f"Edges ({gfci_results['num_edges']}):")
for e in gfci_results["edges"]:
    print(f"  {e}")
print(f"\nDirected: {gfci_graph['directed_edges']}")
print(f"Bidirected: {gfci_graph['bidirected_edges']}")
print(f"Partially oriented: {gfci_graph['partially_oriented_edges']}")
print(f"Circle: {gfci_graph['circle_edges']}")

## Comparison

Since our data has a latent confounder **L** affecting both **X** and **W**:
- **PC** and **FGES** (causal sufficiency assumed) will likely place a direct edge between X and W
- **GFCI** should detect the latent confounding, showing a bidirected edge `X <-> W` or partially oriented edges instead

In [ ]:
print("=" * 50)
print("Algorithm Comparison")
print("=" * 50)
print(f"\n{'Algorithm':<10} {'Edges':>5}  Edge List")
print("-" * 50)
for name, results in [("PC", pc_results), ("FGES", fges_results), ("GFCI", gfci_results)]:
    print(f"{name:<10} {results['num_edges']:>5}  {', '.join(results['edges'])}")